# Phase 4b: Train Steering Vectors on Qwen3.5-35B-A3B-Base

**Input**: `steering_data_n15.json` from Phase 4a (per-cluster training examples).

**Goal**: train 15 category-specific steering vectors + 1 bias vector, following Venhoff et al. 2025 Appendix C.1:
- Add v ∈ ℝ²⁰⁴⁸ to base-model L15 residual stream (at ~37% depth)
- CE loss on target sentence tokens (teacher-forced)
- lr=1e-2 cosine, bs=6, up to 50 iters, patience 5
- Top-2048 examples per cluster by SAE activation (paper filters by base perplexity; we skip for v1)

**Output**: 16 vectors saved to HF repo `qwen35-a3b-sae-phase2/steering_vectors/`.

**Budget**: ~6-7h on RTX 6000 Blackwell 96GB, ~$30-40.

## Cell 1 — Install

In [ ]:
import sys, subprocess, os, shutil
from pathlib import Path

def pip(*a, check=True):
    return subprocess.run([sys.executable, '-m', 'pip', *a], check=check)

try:
    import transformers
    from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
    ok = 'qwen3_5_moe' in CONFIG_MAPPING_NAMES
except Exception:
    ok = False

if not ok:
    pip('install', '-q', 'accelerate', 'datasets', 'huggingface_hub==1.5.0',
        'safetensors', 'einops', 'sentencepiece', 'tokenizers',
        'protobuf', 'scipy', 'hf_transfer', check=False)
    pip('uninstall', '-y', 'transformers', 'causal-conv1d', check=False)
    SRC = '/content/transformers_src'
    if Path(SRC).exists(): shutil.rmtree(SRC)
    subprocess.run(['git','clone','--quiet','--depth=1',
                    'https://github.com/huggingface/transformers.git', SRC], check=True)
    pip('install','--force-reinstall','--no-deps','--no-cache-dir', SRC, check=False)
    pip('install','--no-cache-dir','flash-linear-attention', check=False)
    for m in list(sys.modules):
        if m.startswith('transformers') or m.startswith('huggingface_hub'):
            del sys.modules[m]

try:
    from google.colab import userdata
    t = userdata.get('HF_TOKEN')
    if t:
        from huggingface_hub import login
        login(token=t); print('HF auth OK')
except: pass

import json, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

OUT = Path('/content/phase4b'); OUT.mkdir(exist_ok=True)
HF_PHASE2 = 'caiovicentino1/qwen35-a3b-sae-phase2'
MODEL_ID = 'Qwen/Qwen3.5-35B-A3B-Base'
STEERING_LAYER = 15  # ~37% depth for 40-layer model
N_LATENTS = 15
print('setup done')

## Cell 2 — Load Qwen3.5-35B-A3B-BASE (note: NOT thinking)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import snapshot_download, HfApi

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tok.pad_token_id is None: tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map='cuda',
    attn_implementation='sdpa', trust_remote_code=True)
model.eval()
# Freeze all params (only steering vector gets gradients)
for p in model.parameters(): p.requires_grad_(False)
print(f'Model loaded: {torch.cuda.memory_allocated()/1e9:.1f} GB')

def get_layer_module(m, idx):
    cands = [m]
    if hasattr(m, 'model'): cands.append(m.model)
    for start in cands:
        for path in [('model','language_model','layers'),('language_model','layers'),('model','layers')]:
            cur = start; ok = True
            for p in path:
                if hasattr(cur, p): cur = getattr(cur, p)
                else: ok = False; break
            if ok and hasattr(cur, '__getitem__'):
                try: return cur[idx]
                except: continue
    raise RuntimeError(f'layer {idx} not found')

D = 2048  # hidden size

## Cell 3 — Load steering data + steering hook

In [ ]:
path = snapshot_download(HF_PHASE2, repo_type='model', cache_dir='/content/cache')
with open(Path(path) / f'steering_data_n{N_LATENTS}.json') as f:
    steering_data = json.load(f)
with open(Path(path) / f'cluster_labels_n{N_LATENTS}.json') as f:
    cluster_labels = json.load(f)
print(f'Clusters: {len(steering_data)}')

class SteeringHook:
    """Add steering vector v to L{STEERING_LAYER} residual at every token.
    v is trainable. Hook is forward_pre_hook on the layer block."""
    def __init__(self, dim):
        self.v = nn.Parameter(torch.zeros(dim, device='cuda', dtype=torch.bfloat16))
        self.bias_v = None  # optional frozen bias added during training
        self.active = False
    def on(self):
        self.active = True
    def off(self):
        self.active = False
    def make_hook(self):
        def hook(module, inp):
            if not self.active: return None
            h = inp[0]  # [B, T, d]
            # Add v at every token
            delta = self.v.to(h.dtype)
            if self.bias_v is not None:
                delta = delta + self.bias_v.to(h.dtype)
            h = h + delta
            return (h,) + inp[1:]
        return hook

steering_hook = SteeringHook(D)
layer_module = get_layer_module(model, STEERING_LAYER)
h_handle = layer_module.register_forward_pre_hook(steering_hook.make_hook())
print(f'✅ steering hook installed on L{STEERING_LAYER}')

## Cell 4 — Tokenize + train loop per cluster

In [ ]:
MAX_SEQ_LEN = 2048
BATCH_SIZE = 6
N_ITERS = 50
PATIENCE = 5
DELTA = 0.01
LR = 1e-2
TOP_PER_CLUSTER = 2048  # paper uses 2048 filtered by base-perplexity; we use top-2048 by SAE act

def prepare_example(entry):
    """Build (input_ids, labels) for a training example.
    Labels: -100 for prompt+prefix (masked), target tokens only."""
    prompt_text = entry['prompt']
    prefix_text = entry['prefix']
    target_text = entry['target']
    # Full context: prompt + space + prefix + space + target
    if prefix_text:
        context = f"{prompt_text} {prefix_text}"
    else:
        context = prompt_text
    full = f"{context} {target_text}"
    # Tokenize context and full
    ctx_ids = tok(context, add_special_tokens=True, return_tensors='pt').input_ids[0]
    full_ids = tok(full, add_special_tokens=True, return_tensors='pt').input_ids[0]
    if full_ids.shape[0] > MAX_SEQ_LEN or full_ids.shape[0] <= ctx_ids.shape[0]:
        return None
    labels = full_ids.clone()
    labels[:ctx_ids.shape[0]] = -100  # mask prompt + prefix
    return dict(input_ids=full_ids, labels=labels)

def train_steering_vector(cluster_id, entries):
    # Take top-TOP_PER_CLUSTER by sae_activation (already sorted)
    entries = entries[:TOP_PER_CLUSTER]
    # Pre-tokenize
    examples = []
    for e in entries:
        ex = prepare_example(e)
        if ex is not None: examples.append(ex)
    if len(examples) < BATCH_SIZE:
        print(f'  cluster {cluster_id}: too few examples ({len(examples)}), skip'); return None
    print(f'  cluster {cluster_id}: {len(examples)} tokenized examples')
    
    # Reset vector
    with torch.no_grad():
        steering_hook.v.data.zero_()
    opt = torch.optim.Adam([steering_hook.v], lr=LR)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=N_ITERS)
    
    best_loss = float('inf'); wait = 0; losses = []
    steering_hook.on()
    for it in range(N_ITERS):
        random.shuffle(examples)
        it_losses = []
        for i in range(0, len(examples), BATCH_SIZE):
            batch = examples[i:i+BATCH_SIZE]
            # Pad
            max_len = max(e['input_ids'].shape[0] for e in batch)
            input_ids = torch.full((len(batch), max_len), tok.pad_token_id, dtype=torch.long)
            labels = torch.full((len(batch), max_len), -100, dtype=torch.long)
            attn_mask = torch.zeros((len(batch), max_len), dtype=torch.long)
            for j, e in enumerate(batch):
                L = e['input_ids'].shape[0]
                input_ids[j, :L] = e['input_ids']
                labels[j, :L] = e['labels']
                attn_mask[j, :L] = 1
            input_ids = input_ids.cuda(); labels = labels.cuda(); attn_mask = attn_mask.cuda()
            try:
                out = model(input_ids=input_ids, attention_mask=attn_mask, labels=labels)
                loss = out.loss
                opt.zero_grad()
                loss.backward()
                opt.step()
                it_losses.append(loss.item())
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache(); continue
        scheduler.step()
        avg = np.mean(it_losses) if it_losses else float('inf')
        losses.append(avg)
        if avg < best_loss - DELTA:
            best_loss = avg; wait = 0
            best_v = steering_hook.v.detach().clone().cpu()
        else:
            wait += 1
            if wait >= PATIENCE:
                print(f'  early stop at iter {it+1}, best loss {best_loss:.4f}')
                break
        if (it + 1) % 5 == 0:
            print(f'  iter {it+1}/{N_ITERS}: loss {avg:.4f} (best {best_loss:.4f})')
    steering_hook.off()
    return best_v, losses

print('✅ training function ready')

## Cell 5 — Train all 15 category vectors

In [ ]:
steering_vectors = {}
t0 = time.time()
for cid in range(N_LATENTS):
    cid_str = str(cid)
    label = cluster_labels[cid_str]['title']
    print(f'\n=== Cluster {cid}: {label} ===')
    entries = steering_data[cid_str]
    if not entries:
        print('  empty, skip'); continue
    result = train_steering_vector(cid, entries)
    if result is None: continue
    v, losses = result
    steering_vectors[cid] = dict(vector=v.numpy(), losses=losses, label=label)
    # Save incremental
    np.savez(OUT / f'steering_v_c{cid}.npz', vector=v.numpy(), losses=np.array(losses))
    elapsed = (time.time() - t0) / 60
    print(f'  ✅ cluster {cid} done, {elapsed:.1f} min total')

print(f'\n✅ ALL CLUSTERS DONE in {(time.time()-t0)/60:.1f} min')

## Cell 6 — Train bias vector (on random thinking rollouts)

In [ ]:
# Bias vector: same training procedure, random examples across all clusters
all_entries = []
for cid_str, entries in steering_data.items():
    all_entries.extend(entries[:500])  # take 500 from each cluster = 7500 total
random.seed(42)
random.shuffle(all_entries)
bias_entries = all_entries[:TOP_PER_CLUSTER]
print(f'Training bias vector on {len(bias_entries)} random examples')

print('\n=== Bias vector (no category) ===')
result = train_steering_vector(-1, bias_entries)  # -1 signals bias
if result:
    bias_v, bias_losses = result
    np.savez(OUT / 'steering_bias.npz', vector=bias_v.numpy(), losses=np.array(bias_losses))
    steering_vectors['bias'] = dict(vector=bias_v.numpy(), losses=bias_losses)
    print(f'✅ bias vector trained')

## Cell 7 — Upload to HF

In [ ]:
api = HfApi()

# Upload each vector individually
for cid in range(N_LATENTS):
    path = OUT / f'steering_v_c{cid}.npz'
    if not path.exists(): continue
    try:
        api.upload_file(path_or_fileobj=str(path),
                        path_in_repo=f'steering_vectors/v_c{cid}.npz',
                        repo_id=HF_PHASE2, repo_type='model',
                        commit_message=f'Phase 4b: steering vector cluster {cid}')
    except Exception as e:
        print(f'c{cid}: {e}')

# Bias
bias_path = OUT / 'steering_bias.npz'
if bias_path.exists():
    api.upload_file(path_or_fileobj=str(bias_path),
                    path_in_repo='steering_vectors/bias.npz',
                    repo_id=HF_PHASE2, repo_type='model',
                    commit_message='Phase 4b: bias vector')

# Summary
summary = {}
for cid, v in steering_vectors.items():
    summary[str(cid)] = {
        'label': v.get('label', 'bias'),
        'norm': float(np.linalg.norm(v['vector'])),
        'final_loss': float(v['losses'][-1]) if v['losses'] else None,
        'best_loss': float(min(v['losses'])) if v['losses'] else None,
    }
with open(OUT / 'steering_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
api.upload_file(path_or_fileobj=str(OUT / 'steering_summary.json'),
                path_in_repo='steering_vectors/summary.json',
                repo_id=HF_PHASE2, repo_type='model')

print(f'\n=== Summary ===')
for k, v in summary.items():
    print(f'  {k} ({v["label"]}): norm {v["norm"]:.3f}, loss {v["best_loss"]}')
print(f'\n✅ all vectors uploaded to https://huggingface.co/{HF_PHASE2}/tree/main/steering_vectors')